![torchlight](https://raw.githubusercontent.com/samadpls/torchlight/main/assets/hero.png)

# Fine-tune Qwen3-8B on PyTorch forum answers

This notebook trains [torchlight-qwen3-8b](https://huggingface.co/samadpls/torchlight-qwen3-8b), a PyTorch helper, on one free T4 GPU in about 4 hours.

It learns from [torchlight-threads](https://huggingface.co/datasets/samadpls/torchlight-threads): 6,841 questions from the [PyTorch forums](https://discuss.pytorch.org), each paired with the answer that solved it. The model is trained to give the fix the forum found, and to ask for the code or the error message when a question is missing them.

**Links**

- Model: [torchlight-qwen3-8b](https://huggingface.co/samadpls/torchlight-qwen3-8b), also as a [LoRA adapter](https://huggingface.co/samadpls/torchlight-qwen3-8b-lora) and [GGUF files](https://huggingface.co/samadpls/torchlight-qwen3-8b-GGUF)
- Dataset: [torchlight-threads](https://huggingface.co/datasets/samadpls/torchlight-threads)
- Test answers: [the answers notebook](https://www.kaggle.com/code/samadpls/torchlight-qwen3-8b-answers)
- All the code: [github.com/samadpls/torchlight](https://github.com/samadpls/torchlight)

## What it does

1. Loads Qwen3-8B in 4-bit with [Unsloth](https://github.com/unslothai/unsloth).
2. Turns each forum chat into training text, with Qwen3's thinking mode off.
3. Trains a LoRA adapter for one pass over the data.
4. Checks the validation loss before and after, and asks the model one question.
5. Uploads the adapter, the full model and GGUF files to Hugging Face.

## Before you run it

- In the notebook settings, turn on **GPU T4** and **Internet**. Kaggle asks for a verified phone number for both.
- Add a Hugging Face token with write access as a secret named `HF_API` (**Add-ons → Secrets**).
- Change `HUB_REPO` below to a repo under your own account.
- Start it with **Save Version → Save & Run All**. Kaggle gives secrets only to runs started from this page.

Set `SMOKE = True` for a 10-minute test of every step that uploads nothing.

## Settings

In [ ]:
import json
import os

from kaggle_secrets import UserSecretsClient

SMOKE = False
BASE_MODEL = "unsloth/Qwen3-8B"
DATASET = "samadpls/torchlight-threads"
HUB_REPO = "samadpls/torchlight-qwen3-8b"
MAX_LENGTH = 4096
OUT = "/kaggle/working"
SYSTEM_PROMPT = (
    "You are a careful PyTorch helper. Give specific, correct guidance with code where it helps. "
    "If a question lacks the details needed to answer it, ask for them, for example a minimal "
    "executable code snippet or the full error message, instead of guessing."
)

# Unsloth trains on one GPU; this must be set before torch is imported.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Fail now, not after hours of training, if the secret isn't available. The smoke run
# publishes nothing, so it skips it.
HF_TOKEN = None if SMOKE else UserSecretsClient().get_secret("HF_API")

In [ ]:
%pip install -q unsloth

## Load the model

Qwen3-8B in 4-bit, with LoRA (rank 16, alpha 32) on every attention and MLP layer. About 0.5% of the weights are trained.

In [ ]:
# Unsloth patches transformers, so it is imported first.
from unsloth import FastLanguageModel  # isort: skip

import torch
from datasets import load_dataset
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_MODEL, max_seq_length=MAX_LENGTH, load_in_4bit=True
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=0,
)

## Prepare the chats

Each row is a chat: the question, then the answer that solved it. Only the answer is trained, not the question. Chats longer than 4,096 tokens are left out.

In [ ]:
def tokenize(row):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + row["messages"]
    # With thinking off, Qwen3's prompt ends in an empty <think></think> block; the answer
    # follows it, and only the answer and its end token are trained.
    prompt = tokenizer.apply_chat_template(
        messages[:-1], add_generation_prompt=True, tokenize=False, enable_thinking=False
    )
    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
    answer = messages[-1]["content"] + "<|im_end|>\n"
    answer_ids = tokenizer(answer, add_special_tokens=False).input_ids
    ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}


def fits(row):
    return len(row["input_ids"]) <= MAX_LENGTH


sft = load_dataset(DATASET, "sft")
columns = sft["train"].column_names
train = sft["train"].map(tokenize, remove_columns=columns).filter(fits)
valid = sft["validation"].map(tokenize, remove_columns=columns).filter(fits)
if SMOKE:
    train, valid = train.select(range(40)), valid.select(range(10))
print(f"train: {len(train)} chats, validation: {len(valid)} chats")

## Train

One pass over the data, 8 chats per step (2 at a time, 4 steps of gradient accumulation), with a learning rate of 2e-4 on a cosine schedule. The validation loss is checked once before training and every 200 steps.

In [ ]:
trainer = Trainer(
    model=model,
    train_dataset=train,
    eval_dataset=valid,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True),
    args=TrainingArguments(
        output_dir=f"{OUT}/checkpoints",
        num_train_epochs=1,
        max_steps=5 if SMOKE else -1,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        # transformers 5 reads a float below 1 as a share of the steps.
        warmup_steps=0.05,
        weight_decay=0.01,
        optim="adamw_8bit",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        train_sampling_strategy="group_by_length",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=5 if SMOKE else 200,
        save_strategy="steps",
        save_steps=5 if SMOKE else 200,
        save_total_limit=1,
        seed=0,
        report_to="none",
    ),
)
base_eval = trainer.evaluate()
print("validation loss before training:", base_eval["eval_loss"])
trainer.train()
log = [{"step": 0, "eval_loss": base_eval["eval_loss"]}] + trainer.state.log_history
with open(f"{OUT}/log_history.json", "w") as f:
    json.dump(log, f, indent=1)

## Ask it a question

In [ ]:
FastLanguageModel.for_inference(model)
question = "How do I check which device a tensor is on?"
chat = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
text = tokenizer.apply_chat_template(
    chat, add_generation_prompt=True, tokenize=False, enable_thinking=False
)
inputs = tokenizer(text, return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
answer = output[0][inputs.input_ids.shape[1] :]
print("sample answer:", tokenizer.decode(answer, skip_special_tokens=True))

model.save_pretrained(f"{OUT}/adapter")
tokenizer.save_pretrained(f"{OUT}/adapter")

## Upload to Hugging Face

The smoke run skips this. If the GGUF export fails, the model is already uploaded.

In [ ]:
if SMOKE:
    print("smoke run done, nothing uploaded")
else:
    model.push_to_hub(f"{HUB_REPO}-lora", token=HF_TOKEN)
    tokenizer.push_to_hub(f"{HUB_REPO}-lora", token=HF_TOKEN)
    print("pushed the adapter")
    model.push_to_hub_merged(HUB_REPO, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
    print("pushed the merged model")
    try:
        model.push_to_hub_gguf(
            f"{HUB_REPO}-GGUF", tokenizer, quantization_method=["q4_k_m", "q8_0"], token=HF_TOKEN
        )
        print("pushed the GGUF files")
    except Exception as error:
        print("GGUF export failed:", repr(error))

## Results

The published model came from the training run in this notebook's version history: 863 steps and 4.1 hours of training on one T4. Open that version to see its full log. The validation loss fell from 3.059 to 1.532.

![Training and validation loss](https://raw.githubusercontent.com/samadpls/torchlight/main/assets/training-loss-qwen3-8b.png)

The model was then tested on 163 forum questions asked from April 2025 on, after Qwen3's training data ends. Claude Opus 5.5, as judge, graded each answer from 1 to 5 against the post that solved the thread. The baseline is plain Qwen3-8B on the same questions.

| Model | Average grade (out of 5) | Graded 4 or 5 | Wrong answers (graded 1) |
|---|---|---|---|
| Qwen3-8B (base model) | 2.17 | 17% | 31% |
| torchlight-qwen3-8b (fine-tuned) | 2.25 | 9% | 15% |

Fine-tuning cut the wrong answers in half, and the model asks for the code or the error when a question is missing them. Plain Qwen3-8B gives more complete fixes, mostly because it writes much longer answers. The [evaluation](https://github.com/samadpls/torchlight/tree/main/src/torchlight/eval) has the details.

## Credits

Made by [Abdul Samad](https://samadpls.com) (abdulsamadsid1@gmail.com). The answers come from the PyTorch forum community. The dataset is licensed CC BY 4.0, and the model Apache-2.0, like Qwen3-8B. Everything else, from the forum crawl to the evaluation, is on [GitHub](https://github.com/samadpls/torchlight).